In [1]:
import sys
sys.path.append("/home/jovyan/work")

from src.utils.lakehouse import read_table, write_table
from src.utils.spark_session import get_spark_session
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType, StringType,
    DateType, TimestampType, BooleanType, DecimalType
)
from delta.tables import DeltaTable
from datetime import date, datetime, timedelta
from decimal import Decimal

spark = get_spark_session("build-gold-layer")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

## Write Dimension and Fact Tables

In [2]:
# dim_date

start_date = date(2025, 1, 1)
end_date = date(2027, 12, 31)

date_schema = StructType([
    StructField("date_key", IntegerType(), False),
    StructField("full_date", DateType(), False),
    StructField("day_of_week", StringType(), True),
    StructField("week_of_year", IntegerType(), True),
    StructField("month_no", IntegerType(), True),
    StructField("month_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("is_weekend", BooleanType(), True),
])

date_rows = []
d = start_date
while d <= end_date:
    date_rows.append(Row(
        date_key=int(d.strftime("%Y%m%d")),
        full_date=d,
        day_of_week=d.strftime("%A"),
        week_of_year=d.isocalendar()[1],
        month_no=d.month,
        month_name=d.strftime("%B"),
        quarter=(d.month - 1) // 3 + 1,
        year=d.year,
        is_weekend=d.weekday() >= 5,
    ))
    d += timedelta(days=1)

df_dim_date = spark.createDataFrame(date_rows, schema=date_schema)
df_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")

print(f"gold.dim_date: {spark.table('gold.dim_date').count()} rows")
spark.table("gold.dim_date").orderBy("full_date").limit(3).toPandas()

gold.dim_date: 1095 rows


,date_key,full_date,day_of_week,week_of_year,month_no,month_name,quarter,year,is_weekend
0,20250101,2025-01-01,Wednesday,1,1,January,1,2025,False
1,20250102,2025-01-02,Thursday,1,1,January,1,2025,False
2,20250103,2025-01-03,Friday,1,1,January,1,2025,False


In [4]:
# dim_vehicle

vehicle_rows = read_table(spark, "silver", "vehicle").filter("is_deleted = false").select(
    "vehicle_id", "vin", "make", "model", "model_year", "branch_id",
    F.col("current_stage").alias("stage"),
    F.col("current_status").alias("status"),
    "created_at",
).collect()

dim_vehicle_schema = StructType([
    StructField("vehicle_key", LongType(), False),
    StructField("vehicle_id", LongType(), False),
    StructField("vin", StringType(), True),
    StructField("make", StringType(), True),
    StructField("model", StringType(), True),
    StructField("model_year", IntegerType(), True),
    StructField("branch_id", LongType(), True),
    StructField("stage", StringType(), True),
    StructField("status", StringType(), True),
    StructField("effective_from", TimestampType(), False),
    StructField("effective_to", TimestampType(), True),
    StructField("is_current", BooleanType(), False),
])

dim_vehicle_rows = [
    Row(
        vehicle_key=i, vehicle_id=row.vehicle_id,
        vin=row.vin, make=row.make, model=row.model, model_year=row.model_year,
        branch_id=row.branch_id, stage=row.stage, status=row.status,
        effective_from=row.created_at, effective_to=None, is_current=True,
    )
    for i, row in enumerate(vehicle_rows, start=1)
]

df_dim_vehicle = spark.createDataFrame(dim_vehicle_rows, schema=dim_vehicle_schema)
# df_dim_vehicle.write.format("delta").mode("overwrite").saveAsTable("gold.dim_vehicle")
write_table(df_dim_vehicle, "gold", "dim_vehicle")

print(f"gold.dim_vehicle: {read_table(spark, 'gold', 'dim_vehicle').count()} rows")
read_table(spark, "gold", "dim_vehicle").groupBy("stage", "status").count().show()

gold.dim_vehicle: 40 rows
+----------+------+-----+
|     stage|status|count|
+----------+------+-----+
|   AT_PORT|ACTIVE|    4|
| IN_REPAIR|ACTIVE|    6|
|   SHIPPED|ACTIVE|    4|
|      SOLD|  SOLD|   15|
|    LISTED|ACTIVE|    9|
|INSPECTING|ACTIVE|    2|
+----------+------+-----+



In [5]:
# dim branch, staff, auction_house, vendor, part, customer, data_source, expense_category, payment_method

from pyspark.sql import Window

def add_surrogate_key(df, key_name, order_col):
    return df.withColumn(key_name, F.row_number().over(Window.orderBy(order_col)))

df_dim_branch = add_surrogate_key(
    read_table(spark, "silver", "branch").select("branch_id", "name", "is_active"),
    "branch_key", "branch_id")
write_table(df_dim_branch, "gold", "dim_branch")

df_roles = read_table(spark, "silver", "staff_role").groupBy("staff_id").agg(F.min("role_code").alias("primary_role"))
df_staff_full = read_table(spark, "silver", "staff").select("staff_id", "name", "branch_id").join(df_roles, "staff_id", "left")
df_dim_staff = add_surrogate_key(df_staff_full, "staff_key", "staff_id") \
    .withColumn("effective_from", F.current_timestamp()) \
    .withColumn("effective_to", F.lit(None).cast("timestamp")) \
    .withColumn("is_current", F.lit(True))
write_table(df_dim_staff, "gold", "dim_staff")

df_dim_auction_house = add_surrogate_key(
    read_table(spark, "silver", "auction_house").select("auction_house_id", "name", "country"),
    "auction_house_key", "auction_house_id")
write_table(df_dim_auction_house, "gold", "dim_auction_house")

df_dim_vendor = add_surrogate_key(
    read_table(spark, "silver", "vendor").select("vendor_id", "name"),
    "vendor_key", "vendor_id")
write_table(df_dim_vendor, "gold", "dim_vendor")

df_dim_part = add_surrogate_key(
    read_table(spark, "silver", "part").select("part_id", "name", "category"),
    "part_key", "part_id")
write_table(df_dim_part, "gold", "dim_part")

df_dim_customer = add_surrogate_key(
    read_table(spark, "silver", "customer").select("customer_id", "name", "phone"),
    "customer_key", "customer_id")
write_table(df_dim_customer, "gold", "dim_customer")

df_dim_data_source = add_surrogate_key(
    read_table(spark, "silver", "data_source").select("source_id", "source_code"),
    "source_key", "source_id")
write_table(df_dim_data_source, "gold", "dim_data_source")

df_dim_expense_category = spark.createDataFrame(
    [Row(category_key=i, category_code=c) for i, c in enumerate(["CUSTOMS", "TRANSPORT", "REPAIR", "MISC"], start=1)])
write_table(df_dim_expense_category, "gold", "dim_expense_category")

df_dim_payment_method = spark.createDataFrame(
    [Row(method_key=i, method_code=c) for i, c in enumerate(["BANK_TRANSFER", "POS", "CASH"], start=1)])
write_table(df_dim_payment_method, "gold", "dim_payment_method")

for t in ["dim_branch", "dim_staff", "dim_auction_house", "dim_vendor", "dim_part",
          "dim_customer", "dim_data_source", "dim_expense_category", "dim_payment_method"]:
    print(f"gold.{t}: {read_table(spark, 'gold', t).count()} rows")


gold.dim_branch: 1 rows
gold.dim_staff: 6 rows
gold.dim_auction_house: 3 rows
gold.dim_vendor: 3 rows
gold.dim_part: 8 rows
gold.dim_customer: 15 rows
gold.dim_data_source: 4 rows
gold.dim_expense_category: 4 rows
gold.dim_payment_method: 3 rows


In [6]:
# Fact Purchase

FX_RATE = Decimal("1600.00")

df_purchase = read_table(spark, "silver", "purchase")
df_vehicle_silver = read_table(spark, "silver", "vehicle").select("vehicle_id", "branch_id")
df_dim_vehicle_cur = read_table(spark, "gold", "dim_vehicle").filter("is_current = true").select("vehicle_key", "vehicle_id")
df_dim_auction = read_table(spark, "gold", "dim_auction_house").select("auction_house_key", "auction_house_id")
df_dim_staff_cur = read_table(spark, "gold", "dim_staff").filter("is_current = true").select("staff_key", "staff_id")
df_dim_branch = read_table(spark, "gold", "dim_branch").select("branch_key", "branch_id")
df_dim_source = read_table(spark, "gold", "dim_data_source").select("source_key", "source_id")

df_fact_purchase = (
    df_purchase
    .join(df_vehicle_silver, "vehicle_id")
    .join(df_dim_vehicle_cur, "vehicle_id")
    .join(df_dim_auction, "auction_house_id", "left")
    .join(df_dim_staff_cur, df_purchase.buyer_staff_id == df_dim_staff_cur.staff_id, "left")
    .join(df_dim_branch, "branch_id", "left")
    .join(df_dim_source, "source_id", "left")
    .withColumn("purchase_date_key", F.date_format("purchase_date", "yyyyMMdd").cast("int"))
    .withColumn("fx_rate_used", F.lit(float(FX_RATE)))
    .withColumn("price_amount_usd", F.col("price_amount"))
    .withColumn("price_amount_ngn", F.col("price_amount") * F.lit(float(FX_RATE)))
    .withColumnRenamed("staff_key", "buyer_staff_key")
    .select(
        "vehicle_key", "auction_house_key", "buyer_staff_key", "branch_key",
        "purchase_date_key", "source_key",
        "price_amount_usd", "price_amount_ngn", "fx_rate_used",
    )
)

write_table(df_fact_purchase, "gold", "fact_purchase")
print(f"gold.fact_purchase: {read_table(spark, 'gold', 'fact_purchase').count()} rows")
read_table(spark, "gold", "fact_purchase").limit(5).toPandas()

gold.fact_purchase: 40 rows


,vehicle_key,auction_house_key,buyer_staff_key,branch_key,purchase_date_key,source_key,price_amount_usd,price_amount_ngn,fx_rate_used
0,3,1,1,1,20260322,1,2571.54,4114464.0,1600.0
1,1,3,1,1,20260224,1,5542.33,8867728.0,1600.0
2,4,3,1,1,20260416,1,3144.91,5031856.0,1600.0
3,5,1,1,1,20260624,1,7168.37,11469392.0,1600.0
4,2,3,1,1,20260607,1,7440.60,11904960.0,1600.0


In [7]:
# Fact repair

df_repair_job_part = read_table(spark, "silver", "repair_job_part")
df_repair_job = read_table(spark, "silver", "repair_job").select(
    "repair_job_id", "vehicle_id", "mechanic_staff_id", "start_date", "end_date", "source_id"
)
df_dim_vendor = read_table(spark, "gold", "dim_vendor").select("vendor_key", "vendor_id")
df_dim_part = read_table(spark, "gold", "dim_part").select("part_key", "part_id")

df_fact_repair = (
    df_repair_job_part
    .join(df_repair_job, "repair_job_id")
    .join(df_vehicle_silver, "vehicle_id")
    .join(df_dim_vehicle_cur, "vehicle_id")
    .join(df_dim_staff_cur, df_repair_job.mechanic_staff_id == df_dim_staff_cur.staff_id, "left")
    .join(df_dim_part, "part_id", "left")
    .join(df_dim_vendor, "vendor_id", "left")
    .join(df_dim_branch, "branch_id", "left")
    .join(df_dim_source, "source_id", "left")
    .withColumn("repair_start_date_key", F.date_format("start_date", "yyyyMMdd").cast("int"))
    .withColumn("repair_end_date_key", F.date_format("end_date", "yyyyMMdd").cast("int"))
    .withColumn("line_cost_ngn", F.col("unit_cost") * F.col("qty"))
    .withColumnRenamed("staff_key", "mechanic_staff_key")
    .withColumnRenamed("unit_cost", "unit_cost_ngn")
    .select(
        "vehicle_key", "mechanic_staff_key", "part_key", "vendor_key", "branch_key",
        "repair_start_date_key", "repair_end_date_key", "source_key",
        "qty", "unit_cost_ngn", "line_cost_ngn",
    )
)

write_table(df_fact_repair, "gold", "fact_repair")
print(f"gold.fact_repair: {read_table(spark, 'gold', 'fact_repair').count()} rows")
read_table(spark, "gold", "fact_repair").limit(5).toPandas()

gold.fact_repair: 35 rows


,vehicle_key,mechanic_staff_key,part_key,vendor_key,branch_key,repair_start_date_key,repair_end_date_key,source_key,qty,unit_cost_ngn,line_cost_ngn
0,35,4,2,2,1,20260517,20260522,1,1,73586.00,73586.00
1,35,4,1,3,1,20260517,20260522,1,1,177138.00,177138.00
2,12,4,2,1,1,20260309,20260317,1,1,25192.00,25192.00
3,15,5,8,3,1,20260628,20260702,1,1,37195.00,37195.00
4,30,4,2,1,1,20260703,20260712,1,1,34401.00,34401.00


In [8]:
df_sale = read_table(spark, "silver", "sale")
df_dim_customer = read_table(spark, "gold", "dim_customer").select("customer_key", "customer_id")

df_fact_sale = (
    df_sale
    .join(df_vehicle_silver, "vehicle_id")
    .join(df_dim_vehicle_cur, "vehicle_id")
    .join(df_dim_customer, "customer_id", "left")
    .join(df_dim_staff_cur, df_sale.salesperson_staff_id == df_dim_staff_cur.staff_id, "left")
    .join(df_dim_branch, "branch_id", "left")
    .join(df_dim_source, "source_id", "left")
    .withColumn("sale_date_key", F.date_format("sale_date", "yyyyMMdd").cast("int"))
    .withColumnRenamed("staff_key", "salesperson_staff_key")
    .withColumnRenamed("agreed_price", "agreed_price_ngn")
    .select(
        "vehicle_key", "customer_key", "salesperson_staff_key", "branch_key",
        "sale_date_key", "source_key", "agreed_price_ngn", "status",
    )
)

write_table(df_fact_sale, "gold", "fact_sale")
print(f"gold.fact_sale: {read_table(spark, 'gold', 'fact_sale').count()} rows")
read_table(spark, "gold", "fact_sale").groupBy("status").count().show()

gold.fact_sale: 15 rows
+---------------+-----+
|         status|count|
+---------------+-----+
|      FINALIZED|   14|
|PENDING_PAYMENT|    1|
+---------------+-----+



In [9]:
# Fact vehicle_lifestyle

purchase_dates = read_table(spark, "silver", "purchase").groupBy("vehicle_id").agg(F.min("purchase_date").alias("purchase_date"))
ship_dates = read_table(spark, "silver", "shipment").groupBy("vehicle_id").agg(F.min("etd").alias("ship_date"))
port_dates = read_table(spark, "silver", "port_arrival").groupBy("vehicle_id").agg(
    F.min("arrival_date").alias("port_arrival_date"),
    F.max(F.when(F.col("clearance_status") == "CLEARED", F.col("cleared_date"))).alias("cleared_date"),
)
pickup_dates = read_table(spark, "silver", "pickup").groupBy("vehicle_id").agg(F.min("pickup_date").alias("pickup_date"))
intake_dates = read_table(spark, "silver", "office_intake").groupBy("vehicle_id").agg(F.min("intake_date").alias("office_intake_date"))
inspection_agg = read_table(spark, "silver", "inspection").groupBy("vehicle_id").agg(
    F.min(F.when(F.col("round_no") == 1, F.col("inspection_date"))).alias("first_inspection_date"),
    F.max(F.when(F.col("overall_result") == "PASS", F.col("inspection_date"))).alias("inspection_pass_date"),
    F.max("round_no").alias("inspection_rounds_count"),
)
repair_agg = read_table(spark, "silver", "repair_job").groupBy("vehicle_id").agg(
    F.min("start_date").alias("repair_start_date"),
    F.max(F.when(F.col("status") == "DONE", F.col("end_date"))).alias("repair_end_date"),
)
cleaning_dates = read_table(spark, "silver", "cleaning_task").filter("status='DONE'").groupBy("vehicle_id").agg(F.min("task_date").alias("cleaning_complete_date"))
listing_dates = read_table(spark, "silver", "listing").groupBy("vehicle_id").agg(F.min("listed_date").alias("listed_date"))
sale_dates = read_table(spark, "silver", "sale").groupBy("vehicle_id").agg(F.min("sale_date").alias("sold_date"))

vehicles_base = read_table(spark, "silver", "vehicle").select("vehicle_id", "current_stage", "updated_at")

df_lifecycle = (
    vehicles_base
    .join(purchase_dates, "vehicle_id", "left")
    .join(ship_dates, "vehicle_id", "left")
    .join(port_dates, "vehicle_id", "left")
    .join(pickup_dates, "vehicle_id", "left")
    .join(intake_dates, "vehicle_id", "left")
    .join(inspection_agg, "vehicle_id", "left")
    .join(repair_agg, "vehicle_id", "left")
    .join(cleaning_dates, "vehicle_id", "left")
    .join(listing_dates, "vehicle_id", "left")
    .join(sale_dates, "vehicle_id", "left")
    .join(df_dim_vehicle_cur, "vehicle_id")
)

def date_key(col):
    return F.when(F.col(col).isNotNull(), F.date_format(col, "yyyyMMdd").cast("int"))

def day_diff(a, b):
    return F.when(F.col(a).isNotNull() & F.col(b).isNotNull(), F.datediff(F.col(b), F.col(a)))

df_fact_lifecycle = (
    df_lifecycle
    .withColumn("purchase_date_key", date_key("purchase_date"))
    .withColumn("ship_date_key", date_key("ship_date"))
    .withColumn("port_arrival_date_key", date_key("port_arrival_date"))
    .withColumn("cleared_date_key", date_key("cleared_date"))
    .withColumn("pickup_date_key", date_key("pickup_date"))
    .withColumn("office_intake_date_key", date_key("office_intake_date"))
    .withColumn("first_inspection_date_key", date_key("first_inspection_date"))
    .withColumn("inspection_pass_date_key", date_key("inspection_pass_date"))
    .withColumn("repair_start_date_key", date_key("repair_start_date"))
    .withColumn("repair_end_date_key", date_key("repair_end_date"))
    .withColumn("cleaning_complete_date_key", date_key("cleaning_complete_date"))
    .withColumn("listed_date_key", date_key("listed_date"))
    .withColumn("sold_date_key", date_key("sold_date"))
    .withColumn("days_purchase_to_port", day_diff("purchase_date", "port_arrival_date"))
    .withColumn("days_port_to_clearance", day_diff("port_arrival_date", "cleared_date"))
    .withColumn("days_clearance_to_pickup", day_diff("cleared_date", "pickup_date"))
    .withColumn("days_pickup_to_intake", day_diff("pickup_date", "office_intake_date"))
    .withColumn("days_intake_to_inspection", day_diff("office_intake_date", "first_inspection_date"))
    .withColumn("days_inspection_to_repair_complete", day_diff("first_inspection_date", "repair_end_date"))
    .withColumn("days_repair_to_cleaning", day_diff("repair_end_date", "cleaning_complete_date"))
    .withColumn("days_cleaning_to_listed", day_diff("cleaning_complete_date", "listed_date"))
    .withColumn("days_listed_to_sold", day_diff("listed_date", "sold_date"))
    .withColumn("total_days_purchase_to_sale", day_diff("purchase_date", "sold_date"))
    .select(
        "vehicle_key",
        "purchase_date_key", "ship_date_key", "port_arrival_date_key", "cleared_date_key",
        "pickup_date_key", "office_intake_date_key", "first_inspection_date_key",
        "inspection_pass_date_key", "inspection_rounds_count",
        "repair_start_date_key", "repair_end_date_key", "cleaning_complete_date_key",
        "listed_date_key", "sold_date_key",
        "days_purchase_to_port", "days_port_to_clearance", "days_clearance_to_pickup",
        "days_pickup_to_intake", "days_intake_to_inspection", "days_inspection_to_repair_complete",
        "days_repair_to_cleaning", "days_cleaning_to_listed", "days_listed_to_sold",
        "total_days_purchase_to_sale",
        "current_stage",
        F.col("updated_at").alias("last_updated_at"),
    )
)

write_table(df_fact_lifecycle, "gold", "fact_vehicle_lifecycle")
print(f"gold.fact_vehicle_lifecycle: {read_table(spark, 'gold', 'fact_vehicle_lifecycle').count()} rows")
read_table(spark, "gold", "fact_vehicle_lifecycle").select(
    "vehicle_key", "current_stage", "total_days_purchase_to_sale"
).orderBy(F.desc("total_days_purchase_to_sale")).limit(10).toPandas()

gold.fact_vehicle_lifecycle: 40 rows


,vehicle_key,current_stage,total_days_purchase_to_sale
0,9,SOLD,126
1,25,SOLD,112
2,16,SOLD,98
3,1,SOLD,92
4,7,SOLD,83
5,8,SOLD,81
6,10,SOLD,79
7,21,SOLD,77
8,15,SOLD,76
9,33,SOLD,76


In [10]:
# Fact profit

df_cost_purchase = read_table(spark, "gold", "fact_purchase").groupBy("vehicle_key").agg(F.sum("price_amount_ngn").alias("purchase_cost_ngn"))
df_cost_repair = read_table(spark, "gold", "fact_repair").groupBy("vehicle_key").agg(F.sum("line_cost_ngn").alias("repair_cost_ngn"))
df_revenue = read_table(spark, "gold", "fact_sale").filter("status IN ('FINALIZED','PENDING_PAYMENT')").groupBy("vehicle_key").agg(F.max("agreed_price_ngn").alias("revenue_ngn"))

df_profit = (
    read_table(spark, "gold", "dim_vehicle").filter("is_current = true").select("vehicle_key")
    .join(df_cost_purchase, "vehicle_key", "left")
    .join(df_cost_repair, "vehicle_key", "left")
    .join(df_revenue, "vehicle_key", "left")
    .withColumn("purchase_cost_ngn", F.coalesce("purchase_cost_ngn", F.lit(0.0)))
    .withColumn("repair_cost_ngn", F.coalesce("repair_cost_ngn", F.lit(0.0)))
    .withColumn("total_cost_ngn", F.col("purchase_cost_ngn") + F.col("repair_cost_ngn"))
    .withColumn("profit_ngn", F.when(F.col("revenue_ngn").isNotNull(), F.col("revenue_ngn") - F.col("total_cost_ngn")))
    .withColumn("margin_pct", F.when(F.col("revenue_ngn").isNotNull() & (F.col("revenue_ngn") != 0),
                                      F.round(F.col("profit_ngn") / F.col("revenue_ngn") * 100, 2)))
    .withColumn("calculated_at", F.current_timestamp())
    .select("vehicle_key", "total_cost_ngn", F.col("revenue_ngn").alias("total_revenue_ngn"), "profit_ngn", "margin_pct", "calculated_at")
)

write_table(df_profit, "gold", "fact_profit")
print(f"gold.fact_profit: {read_table(spark, 'gold', 'fact_profit').count()} rows")
read_table(spark, "gold", "fact_profit").filter("profit_ngn is not null").orderBy(F.desc("profit_ngn")).limit(10).toPandas()

gold.fact_profit: 40 rows


,vehicle_key,total_cost_ngn,total_revenue_ngn,profit_ngn,margin_pct,calculated_at
0,33,12729360.0,18168590.00,5439230.0,29.94,2026-07-20 13:03:41.626943
1,39,12409328.0,16508500.00,4099172.0,24.83,2026-07-20 13:03:41.626943
2,8,9754308.0,13644960.00,3890652.0,28.51,2026-07-20 13:03:41.626943
3,16,10940424.0,14469420.00,3528996.0,24.39,2026-07-20 13:03:41.626943
4,25,7057763.0,9731520.00,2673757.0,27.48,2026-07-20 13:03:41.626943
5,24,8732080.0,11134350.00,2402270.0,21.58,2026-07-20 13:03:41.626943
6,1,8867728.0,11115555.00,2247827.0,20.22,2026-07-20 13:03:41.626943
7,4,5238590.0,7213020.00,1974430.0,27.37,2026-07-20 13:03:41.626943
8,12,8517144.0,10209360.00,1692216.0,16.58,2026-07-20 13:03:41.626943
9,9,4077702.0,5550120.00,1472418.0,26.53,2026-07-20 13:03:41.626943


In [11]:
# fact_profit_history

df_profit_history = (
    read_table(spark, "gold", "fact_profit")
    .withColumn("reason", F.lit("initial calculation"))
    .withColumn("profit_history_key", F.monotonically_increasing_id())
)

write_table(df_profit_history, "gold", "fact_profit_history", mode="append")
print(f"gold.fact_profit_history: {read_table(spark, 'gold', 'fact_profit_history').count()} rows")

gold.fact_profit_history: 40 rows


In [12]:
# Quick check

read_table(spark, "gold", "fact_profit").filter("profit_ngn is not null").agg(
    F.count("*").alias("vehicles_sold"),
    F.sum("total_revenue_ngn").alias("total_revenue_ngn"),
    F.sum("total_cost_ngn").alias("total_cost_ngn"),
    F.sum("profit_ngn").alias("total_profit_ngn"),
    F.round(F.avg("margin_pct"), 2).alias("avg_margin_pct"),
).show(truncate=False)

+-------------+-----------------+--------------+----------------+--------------+
|vehicles_sold|total_revenue_ngn|total_cost_ngn|total_profit_ngn|avg_margin_pct|
+-------------+-----------------+--------------+----------------+--------------+
|15           |145513395.00     |1.10913153E8  |3.4600242E7     |22.91         |
+-------------+-----------------+--------------+----------------+--------------+

